In [ ]:
# Đóng SparkSession khi hoàn thành
# spark.stop()
# print("✅ SparkSession đã đóng")

print("⚠️ Giữ SparkSession mở để tiếp tục phân tích...")

## 10. Close Spark Session

In [ ]:
print("📊 TỔNG KẾT PHÂN TÍCH:")
print("=" * 80)
print(f"🎤 Tổng số nghệ sĩ: {dim_artist.count()}")
print(f"🎵 Tổng số tracks: {fact_performance.select('track_id').distinct().count()}")
print(f"🌍 Tổng số markets: {fact_performance.select('market_id').distinct().count()}")
print(f"📅 Số năm phát hành: {dim_date.select('year').distinct().count()}")

print("\n📈 Top 5 Artists Summary:")
print(top5_pd[['artist_name', 'avg_popularity', 'total_tracks', 'artist_followers']].to_string(index=False))

## 9. Summary Statistics

In [ ]:
# Join fact với dim_date để lấy year
top5_by_year = fact_performance \
    .join(dim_artist, "artist_id") \
    .join(dim_date, "date_id") \
    .groupBy("year", "artist_name") \
    .agg(
        spark_sum("popularity").alias("total_popularity"),
        count("track_id").alias("total_tracks")
    ) \
    .orderBy("year", desc("total_popularity"))

print("📅 TOP CA SĨ THEO TỪNG NĂM:")
print("=" * 80)

# Hiển thị top 5 cho từng năm
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("year").orderBy(desc("total_popularity"))
top5_each_year = top5_by_year.withColumn("rank", row_number().over(window)) \
    .filter(col("rank") <= 5) \
    .orderBy("year", "rank")

top5_each_year.show(50, truncate=False)

## 8. Phân tích theo Năm

Truy vấn Top 5 ca sĩ theo từng năm phát hành.

In [ ]:
# Biểu đồ tròn: Phân bổ Artist Followers
plt.figure(figsize=(10, 10))
plt.pie(top5_pd['artist_followers'], 
        labels=top5_pd['artist_name'], 
        autopct='%1.1f%%',
        startangle=90,
        colors=sns.color_palette('pastel'))
plt.title('👥 Top 5 Artists - Followers Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Biểu đồ tròn đã được tạo!")

In [ ]:
# Biểu đồ cột: Top 5 Artists by Average Popularity
plt.figure(figsize=(12, 6))
plt.barh(top5_pd['artist_name'], top5_pd['avg_popularity'], color='skyblue')
plt.xlabel('Average Popularity', fontsize=12)
plt.ylabel('Artist Name', fontsize=12)
plt.title('🎤 Top 5 Artists by Average Popularity', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()  # Highest at top
for i, v in enumerate(top5_pd['avg_popularity']):
    plt.text(v + 0.5, i, f'{v:.1f}', va='center')
plt.tight_layout()
plt.show()

print("✅ Biểu đồ đã được tạo!")

## 7. Visualize Results

Tạo biểu đồ để hiển thị Top 5 ca sĩ.

In [ ]:
# Cách 2: Tính toán từ FACT table + DIM_ARTIST (join và aggregate)
print("🎤 TOP 5 CA SĨ THEO TỔNG POPULARITY (từ Fact table):")
print("=" * 80)

# Join fact với dim_artist
top5_from_fact = fact_performance \
    .join(dim_artist, "artist_id") \
    .groupBy("artist_name", "artist_popularity", "artist_followers") \
    .agg(
        spark_sum("popularity").alias("total_popularity"),
        avg("popularity").alias("avg_popularity"),
        count("track_id").alias("total_tracks")
    ) \
    .orderBy(desc("total_popularity")) \
    .limit(5)

top5_from_fact.show(truncate=False)

In [ ]:
# Cách 1: Sử dụng bảng AGG_ARTIST_PERFORMANCE (đã tổng hợp sẵn)
print("🎤 TOP 5 CA SĨ CÓ AVG POPULARITY CAO NHẤT:")
print("=" * 80)

top5_artists = agg_artist \
    .select("artist_name", "avg_popularity", "total_tracks", "artist_popularity", "artist_followers") \
    .orderBy(desc("avg_popularity")) \
    .limit(5)

top5_artists.show(truncate=False)

# Convert to pandas for better display
top5_pd = top5_artists.toPandas()
print("\n📊 Kết quả dạng bảng:")
print(top5_pd.to_string(index=False))

## 6. Query Top 5 Artists by Total Popularity

**Lưu ý**: Spotify Web API không cung cấp số lượt nghe (streams) thực tế. Thay vào đó, chúng ta có:
- **popularity**: Chỉ số 0-100 đo lường mức độ phổ biến
- **artist_followers**: Số người theo dõi nghệ sĩ

Ta sẽ phân tích **Top 5 ca sĩ có tổng popularity cao nhất** (tổng tất cả tracks của họ).

In [ ]:
print("📊 FACT_TRACK_PERFORMANCE Schema:")
fact_performance.printSchema()
print("\n📋 Sample data:")
fact_performance.show(5, truncate=False)

In [ ]:
print("📊 DIM_ARTIST Schema:")
dim_artist.printSchema()
print("\n📋 Sample data:")
dim_artist.show(5, truncate=False)

## 5. Explore Data Schema and Sample

Xem cấu trúc dữ liệu để hiểu các cột có sẵn.

In [ ]:
# Đọc DIM_ARTIST
dim_artist = spark.read.csv(DIM_ARTIST_PATH, header=True, inferSchema=True)
print(f"✅ DIM_ARTIST: {dim_artist.count()} records")

# Đọc DIM_DATE  
dim_date = spark.read.csv(DIM_DATE_PATH, header=True, inferSchema=True)
print(f"✅ DIM_DATE: {dim_date.count()} records")

# Đọc FACT_TRACK_PERFORMANCE
fact_performance = spark.read.csv(FACT_PERFORMANCE_PATH, header=True, inferSchema=True)
print(f"✅ FACT_TRACK_PERFORMANCE: {fact_performance.count()} records")

# Đọc AGG_ARTIST_PERFORMANCE (đã tổng hợp sẵn)
agg_artist = spark.read.csv(AGG_ARTIST_PATH, header=True, inferSchema=True)
print(f"✅ AGG_ARTIST_PERFORMANCE: {agg_artist.count()} records")

## 4. Read Data from Gold Bucket

Đọc các bảng từ Gold layer để phân tích.

In [ ]:
spark = SparkSession.builder \
    .appName("Spotify_Gold_Analysis") \
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print(f"✅ SparkSession created: {spark.version}")

## 3. Create Spark Session with MinIO Configuration

In [ ]:
# MinIO Configuration
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "longminh"
MINIO_SECRET_KEY = "longminh"

# Gold bucket paths
GOLD_BUCKET = "gold"
DIM_ARTIST_PATH = f"s3a://{GOLD_BUCKET}/dim_artist"
DIM_DATE_PATH = f"s3a://{GOLD_BUCKET}/dim_date"
FACT_PERFORMANCE_PATH = f"s3a://{GOLD_BUCKET}/fact_track_performance"
AGG_ARTIST_PATH = f"s3a://{GOLD_BUCKET}/agg_artist_performance"

print("📝 Configuration set:")
print(f"   MinIO Endpoint: {MINIO_ENDPOINT}")
print(f"   Gold Bucket: {GOLD_BUCKET}")

## 2. Configure MinIO Connection

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, desc, year
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries imported successfully")

## 1. Import Required Libraries

# 🎵 Phân tích Spotify Data từ Gold Layer

Notebook này sẽ:
1. Kết nối đến MinIO Gold bucket
2. Đọc dữ liệu từ các bảng Dimension và Fact
3. Truy vấn **Top 5 ca sĩ có popularity cao nhất** trong năm
4. Visualize kết quả